# Check Output from Calibration Run to Ensure All Expected Output Files Were Generated

In [1]:
from matplotlib.figure import Figure
from numpy.typing import ArrayLike, NDArray
from pathlib import Path
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
import json
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns
import sqlite3

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

## Check for Missing Files

In [2]:
info(f"Checking outputs for calibration run: {calibration_path}\n")

input_df = pd.read_csv(Path(CALIBRATION_RUN_INPUTS_DIR) / "inputs.csv")
if len(input_df) == 0:
    print("No input files found in 'input' directory. Please ensure that the input files have been generated and are located in the 'input' directory.")
else:
    print(f"Found {len(input_df)} input files.")

output_path = Path(f"{calibration_path}/output")
# output_files = list(output_path.glob("calibration_beta_*_access_*_district_*_pop_*_monthly_data_*.db"))
output_files = [p.name for p in output_path.glob("calibration_beta_*_access_*_district_*_pop_*_monthly_data_*.db")]
print(f"Found {len(output_files)} output files.")

log_path = Path(f"{calibration_path}/log")
# log_files = list(log_path.glob("*.log"))
log_files = [p.name for p in log_path.glob("*.log")]
print(f"Found {len(log_files)} log files.")

script_path = Path(f"{calibration_path}/script")
submit_file = list(script_path.glob("submit_*.pbs"))
if len(submit_file) == 0:
    print("No submit script found in 'script' directory. Please ensure that the submit script has been generated and is located in the 'script' directory.")
else:    
    print(f"Found submit script: {submit_file[0].name}")
cmds_files = [f.name for f in list(script_path.glob("*.txt"))]
print(f"Found {len(cmds_files)} script command files: {', '.join(cmds_files)}")
cmds_lines = []
for cmd_file in cmds_files:
    with open(script_path / cmd_file, "r") as f:
        cmds_lines.extend(f.readlines())


data_path = Path("data")
#Check if data files exist
data_files = [f"{country_code}_pfpr_{calibration_year}.asc", f"{country_code}_treatmentseeking.asc"]
missing_files = [f for f in data_files if not (data_path / f).is_file()]
if missing_files:
    warn(f"Warning: The following data files are missing from the 'data' directory: {', '.join(missing_files)}. Please ensure that these files are present for the analysis to run correctly.")

analysis_path = Path(calibration_analysis_path)
analysis_path.mkdir(exist_ok=True)
print(f"Created analysis directory at {analysis_path.resolve()}")

→ Checking outputs for calibration run: calibration_runs/calibration_1_1_population_scale_one_pattern_25_replicates

Found 570 input files.
Found 14250 output files.
Found 14250 log files.
Found submit script: submit_unfinished.pbs
Found 2 script command files: cmds_LsHARX.txt, cmds_unfinished.txt
⚠ Warning: The following data files are missing from the 'data' directory: ng-se_pfpr_2024.asc, ng-se_treatmentseeking.asc. Please ensure that these files are present for the analysis to run correctly.
Created analysis directory at /work/tuv89272/Calibration_Pipeline_v7_NG-SE/calibration_runs/calibration_1_1_population_scale_one_pattern_25_replicates/analysis


### Display heads of output files, log files, and command lines

In [3]:
print(f"Displaying 5 of {len(output_files)} output files:")
output_files[:5]


Displaying 5 of 14250 output files:


['calibration_beta_0.6_access_0.72_district_11_pop_9158_monthly_data_11.db',
 'calibration_beta_0.015_access_0.731_district_7_pop_1006_monthly_data_20.db',
 'calibration_beta_0.04_access_0.731_district_7_pop_3709_monthly_data_25.db',
 'calibration_beta_0.8_access_0.527_district_32_pop_2380_monthly_data_18.db',
 'calibration_beta_0.001_access_0.731_district_7_pop_40603_monthly_data_22.db']

In [4]:
print(f"Displaying 5 of {len(log_files)} log files:")
log_files[:5]

Displaying 5 of 14250 log files:


['calibration_beta_0.01_access_0.731_district_7_pop_2003_rep_7.log',
 'calibration_beta_0.02_access_0.468_district_35_pop_425_rep_25.log',
 'calibration_beta_0.4_access_0.468_district_35_pop_425_rep_4.log',
 'calibration_beta_0.02_access_0.72_district_11_pop_38975_rep_24.log',
 'calibration_beta_0.015_access_0.731_district_7_pop_8421_rep_1.log']

In [5]:
print(f"Displaying 5 of {len(cmds_lines)} command lines:")
cmds_lines[:5]

Displaying 5 of 28500 command lines:


['./bin/MalaSim -i input/input_beta_0.0010_access_0.527_district_32_pop_987.yml -r SQLiteMonthlyReporter -j 1 -o output/calibration_beta_0.001_access_0.527_district_32_pop_987_ -v 1 > log/calibration_beta_0.001_access_0.527_district_32_pop_987_rep_1.log 2>&1\n',
 './bin/MalaSim -i input/input_beta_0.0010_access_0.527_district_32_pop_987.yml -r SQLiteMonthlyReporter -j 2 -o output/calibration_beta_0.001_access_0.527_district_32_pop_987_ -v 1 > log/calibration_beta_0.001_access_0.527_district_32_pop_987_rep_2.log 2>&1\n',
 './bin/MalaSim -i input/input_beta_0.0010_access_0.527_district_32_pop_987.yml -r SQLiteMonthlyReporter -j 3 -o output/calibration_beta_0.001_access_0.527_district_32_pop_987_ -v 1 > log/calibration_beta_0.001_access_0.527_district_32_pop_987_rep_3.log 2>&1\n',
 './bin/MalaSim -i input/input_beta_0.0010_access_0.527_district_32_pop_987.yml -r SQLiteMonthlyReporter -j 4 -o output/calibration_beta_0.001_access_0.527_district_32_pop_987_ -v 1 > log/calibration_beta_0.001_

### Cross-Check Output Files, Log Files, and Script Command Files

In [6]:
# --- 1. Parse keys from output files ---
# Pattern: calibration_beta_{b}_access_{a}_district_{d}_pop_{p}_monthly_data_{rep}.db
output_re = re.compile(
    r"calibration_beta_([\d.]+)_access_([\d.]+)_district_(\d+)_pop_(\d+)_monthly_data_(\d+)\.db"
)
output_keys = set()
for f in output_files:
    m = output_re.match(f)
    if m:
        output_keys.add(m.groups())  # (beta, access, district, pop, rep)

# --- 2. Parse keys from log files ---
# Pattern: calibration_beta_{b}_access_{a}_district_{d}_pop_{p}_rep_{rep}.log
log_re = re.compile(
    r"calibration_beta_([\d.]+)_access_([\d.]+)_district_(\d+)_pop_(\d+)_rep_(\d+)\.log"
)
log_keys = set()
for f in log_files:
    m = log_re.match(f)
    if m:
        log_keys.add(m.groups())

# --- 3. Parse keys from script command lines (and map key -> original line) ---
# Each line has: ...  > log/calibration_beta_{b}_access_{a}_district_{d}_pop_{p}_rep_{rep}.log ...
cmd_log_re = re.compile(
    r"log/calibration_beta_([\d.]+)_access_([\d.]+)_district_(\d+)_pop_(\d+)_rep_(\d+)\.log"
)
cmd_keys = set()
cmd_key_to_line = {}
for line in cmds_lines:
    stripped = line.strip()
    if not stripped:
        continue
    m = cmd_log_re.search(stripped)
    if m:
        key = m.groups()
        cmd_keys.add(key)
        cmd_key_to_line[key] = stripped

print(f"Unique (beta, access, district, pop, rep) tuples:")
print(f"  Script commands : {len(cmd_keys)}")
print(f"  Log files       : {len(log_keys)}")
print(f"  Output files    : {len(output_keys)}")
print()

# --- 4. Cross-check ---
in_cmd_not_log = cmd_keys - log_keys
in_cmd_not_output = cmd_keys - output_keys
in_log_not_cmd = log_keys - cmd_keys
in_log_not_output = log_keys - output_keys
in_output_not_cmd = output_keys - cmd_keys
in_output_not_log = output_keys - log_keys

all_ok = True

def _report(label, diff_set):
    global all_ok
    if diff_set:
        all_ok = False
        print(f"  {label}: {len(diff_set)} entries")
        for t in sorted(diff_set)[:10]:
            print(f"    beta={t[0]}, access={t[1]}, district={t[2]}, pop={t[3]}, rep={t[4]}")
        if len(diff_set) > 10:
            print(f"    ... and {len(diff_set) - 10} more")

_report("In SCRIPT but missing LOG file", in_cmd_not_log)
_report("In SCRIPT but missing OUTPUT file", in_cmd_not_output)
_report("LOG file exists but not in SCRIPT", in_log_not_cmd)
_report("LOG file exists but no OUTPUT file", in_log_not_output)
_report("OUTPUT file exists but not in SCRIPT", in_output_not_cmd)
_report("OUTPUT file exists but no LOG file", in_output_not_log)

if all_ok:
    print("All checks passed — script commands, log files, and output files are fully consistent.")

# --- 5. Summary by (beta, access, district, pop) ignoring rep ---
strip_rep = lambda s: {t[:4] for t in s}
cmd_configs = strip_rep(cmd_keys)
log_configs = strip_rep(log_keys)
out_configs = strip_rep(output_keys)

print(f"\nUnique configurations (beta, access, district, pop):")
print(f"  Script commands : {len(cmd_configs)}")
print(f"  Log files       : {len(log_configs)}")
print(f"  Output files    : {len(out_configs)}")

missing_output_configs = cmd_configs - out_configs
if missing_output_configs:
    print(f"\n{len(missing_output_configs)} configurations in script with missing output:")
    for t in sorted(missing_output_configs)[:20]:
        print(f"    beta={t[0]}, access={t[1]}, district={t[2]}, pop={t[3]}")
    if len(missing_output_configs) > 20:
        print(f"    ... and {len(missing_output_configs) - 20} more")

# --- 6. Write unfinished commands to file ---
unfinished_keys = sorted((in_cmd_not_output | in_cmd_not_log) & cmd_keys)
unfinished_lines = [cmd_key_to_line[k] for k in unfinished_keys if k in cmd_key_to_line]

unfinished_path = script_path / "cmds_unfinished.txt"
if unfinished_lines:
    with open(unfinished_path, "w") as f:
        f.write("\n".join(unfinished_lines) + "\n")
    warn(f"\nWrote {len(unfinished_lines)} unfinished commands to {unfinished_path}")
else:
    # Remove stale file if everything is now complete
    if unfinished_path.exists():
        unfinished_path.unlink()
    ok(f"\nNo unfinished commands — {unfinished_path.name} not created.")

Unique (beta, access, district, pop, rep) tuples:
  Script commands : 14250
  Log files       : 14250
  Output files    : 14250

All checks passed — script commands, log files, and output files are fully consistent.

Unique configurations (beta, access, district, pop):
  Script commands : 570
  Log files       : 570
  Output files    : 570
✓ 
No unfinished commands — cmds_unfinished.txt not created.


In [7]:
# Make new submist script from found submit_file, just find the cmd_JOBNAME and replace with cmds_unfinished.txt and write to submit_unfinished.pbs
# For example: submit_jobs_lt3ICU.pbs -> submit_unfinished.pbs and inside the submit_unfinished.pbs replace cmds_lt3ICU with cmds_unfinished.txt
if unfinished_lines:
    if submit_file:
        original_submit = submit_file[0]
        new_submit = script_path / "submit_unfinished.pbs"
        with open(original_submit, "r") as f:
            submit_content = f.read()
        # Replace the cmds_*.txt with cmds_unfinished.txt
        modified_content = re.sub(r"cmds_[\w\d]+\.txt", "cmds_unfinished.txt", submit_content)
        with open(new_submit, "w") as f:
            f.write(modified_content)
        warn(f"Created new submit script for unfinished commands: {new_submit}")
else:
    ok(f"No unfinished commands.")

✓ No unfinished commands.
